In [4]:
import matplotlib.pyplot as plt
import time
import numpy as np
import os
import struct
from tqdm import tqdm

CW_SETUP_SCRIPTS = os.environ['CW_SETUP_SCRIPTS']
CW_FIRMWARE_PATH = os.environ['CW_FIRMWARE_PATH']
CW_BASE_PATH     = os.environ['CW_BASE_PATH']
PATH_TO_SETUP    = f'{CW_SETUP_SCRIPTS}/Setup_Generic.ipynb'
SCOPETYPE        = 'CWNANO'
PLATFORM         = 'CWNANO'
SS_VER           = 'SS_VER_1_1'
CRYPTO_TARGET    = 'NONE'


In [5]:
PATH_TO_SETUP = '/home/santourh/Documents/chipwhisperer/Setup_Scripts/Setup_Generic.ipynb'


In [23]:
%%bash -s "$PLATFORM" "$SS_VER" "$CRYPTO_TARGET"
cd ../mayo-src/
make PLATFORM=$1 CRYPTO_TARGET=$3 SS_VER=$2 -j

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWNANO 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
-en     simpleserial-base.c ...
.
.
Compiling:
-en     mayo_core.c ...
Compiling:
Compiling:
-en     mayo_shake.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
.
.
.
Compiling:
Assembling: .././hal//stm32f0/stm32f0_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m0 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWNANO/stm32f0_startup.lst -I. -I.././simpleserial/ -I.././hal/ -I.././hal/ -I.././hal//stm32f0 -I.././hal//stm32f0/CMSIS -I.././hal//stm32f0/CMSIS/core -I.././hal//stm32f0/CMSIS/d

In [6]:
%run "$PATH_TO_SETUP"


(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.65.0) is outdated - latest is 0.66.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


INFO: Found ChipWhisperer😍


In [24]:
import chipwhisperer as cw
prog = cw.programmers.STM32FProgrammer
fw_path = '../mayo-src/simpleserial-base-CWNANO.hex'
cw.program_target(scope, prog, fw_path)


Detected known STMF32: STM32F04xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 9883 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 9883 bytes


In [25]:
import binascii

def mayo_load_key(target, seedsk: bytes):
    """Send 16-byte secret key seed to device ('k' command)."""
    assert len(seedsk) == 16
    target.simpleserial_write('k', seedsk)
    resp = target.simpleserial_read('z', 1, timeout=200)
    return resp

def mayo_sign_capture(scope, target, msg: bytes):
    """Arm scope, send 16-byte message, return (trace, sig_bytes)."""
    assert len(msg) == 16
    scope.arm()
    target.simpleserial_write('p', msg)
    ret = scope.capture()
    if ret:
        print('Capture timeout')
        return None, None
    sig = target.simpleserial_read('r', 66, timeout=5000)
    return scope.get_last_trace(), sig


In [ ]:
# 16-byte secret key seed — change to your test vector
SEEDSK = bytes(range(16))   # 00 01 02 ... 0f
reset_target(scope)
time.sleep(1)
target.simpleserial_write('k', SEEDSK)
resp = target.simpleserial_read_witherrors('z', 1, timeout=500)
print('Key loaded, response:', resp)

(ChipWhisperer Target ERROR|File SimpleSerial.py:351) Target did not ack


Key loaded, response: {'valid': True, 'payload': bytearray(b'\x01'), 'full_response': 'z01\n', 'rv': None}


In [34]:
# Sign a message and read back the signature (no scope capture)
MSG = bytes(range(16))   # 00 01 02 ... 0f

target.simpleserial_write('p', MSG)
sig = target.simpleserial_read('r', 66, timeout=5000)

if sig is None:
    print('ERROR: no response (key loaded?)')
else:
    print(f'Signature ({len(sig)} bytes):')
    print(' ', sig.hex())
    print(f'  s  (first 50 B): {sig[:50].hex()}')
    print(f'  salt (last 16 B): {sig[50:].hex()}')


AttributeError: 'NoneType' object has no attribute 'controlRead'

In [33]:
MSG = bytes(range(16))   # 00 01 02 ... 0f

trace, sig = mayo_sign_capture(scope, target, MSG)
if sig is not None:
    print(f'Signature ({len(sig)} bytes):', sig.hex())
    print(f'Salt (last 16 B):', sig[-16:].hex())
    plt.figure(figsize=(14,3))
    plt.plot(trace)
    plt.title('MAYO-micro signing — power trace')
    plt.xlabel('Sample'); plt.ylabel('ADC value')
    plt.tight_layout(); plt.show()


AttributeError: 'NoneType' object has no attribute 'controlWrite'

In [30]:
N_TRACES = 100
traces = []
sigs   = []

for i in tqdm(range(N_TRACES)):
    msg = np.random.bytes(16)
    t, s = mayo_sign_capture(scope, target, msg)
    if t is not None:
        traces.append(t)
        sigs.append(s)

traces = np.array(traces)
print(f'Captured {len(traces)} traces, shape {traces.shape}')
np.save('mayo_traces.npy', traces)
print('Saved to mayo_traces.npy')


100%|██████████| 100/100 [01:02<00:00,  1.59it/s]

Captured 100 traces, shape (100, 5000)
Saved to mayo_traces.npy


In [31]:
scope.dis()
target.dis()
